<a href="https://colab.research.google.com/github/RoshanHelmy/FlyRank-Assignment1-/blob/main/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


The ranked queue prioritizes content pages for human review based on observed search-performance signals.

The main reason code used in the current baseline is LOW_CTR. This identifies pages receiving impressions but generating relatively few clicks.

The action is PRIORITIZE_FOR_HUMAN_REVIEW. The score determines the order in which pages should be reviewed, rather than automatically changing the content.

The queue is intended as decision support. A high score does not prove that a page needs to be changed.

In [3]:
import os
import pandas as pd
import numpy as np
os.makedirs("work/outputs", exist_ok=True)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded:", bool(HF_TOKEN))
except Exception:
    HF_TOKEN = None
    print("HF_TOKEN not available.")

HF_TOKEN loaded: True


In [4]:
from huggingface_hub import hf_hub_download

repo_id = "FlyRank/internship-warehouse"
performance_path = hf_hub_download(
    repo_id=repo_id,
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

performance = pd.read_parquet(
    performance_path,
    columns=[
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
)

print("Performance loaded:", performance.shape)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Performance loaded: (9841378, 6)


In [5]:

performance["ctr"] = (
    performance["gsc_clicks"] /
    performance["gsc_impressions"].replace(0, np.nan)
).fillna(0)

playbook_df = (
    performance
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        ctr=("ctr", "mean")
    )
)

print("Playbook dataframe:", playbook_df.shape)

display(playbook_df.head())

Playbook dataframe: (331437, 6)


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0


In [6]:
# Rank-based opportunity signals
playbook_df["impressions_rank"] = (
    playbook_df["impressions"].rank(pct=True)
)

playbook_df["ctr_opportunity"] = (
    1 - playbook_df["ctr"].rank(pct=True)
)

playbook_df["position_opportunity"] = (
    playbook_df["avg_position"].rank(pct=True)
)

playbook_df["action_score"] = (
    0.40 * playbook_df["impressions_rank"]
    + 0.30 * playbook_df["ctr_opportunity"]
    + 0.30 * playbook_df["position_opportunity"]
)

In [7]:

playbook_df["reason_code"] = np.select(
    [
        playbook_df["ctr"] <= playbook_df["ctr"].quantile(0.25),
        playbook_df["avg_position"] > 15
    ],
    [
        "LOW_CTR",
        "WEAK_POSITION"
    ],
    default="REVIEW"
)

playbook_df["action"] = "PRIORITIZE_FOR_HUMAN_REVIEW"

# Rank the queue
playbook_df = playbook_df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

playbook_df["rank"] = np.arange(1, len(playbook_df) + 1)

print("Ranked queue created.")
display(
    playbook_df[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "action_score",
            "reason_code",
            "action",
            "impressions",
            "ctr",
            "avg_position"
        ]
    ].head(10)
)

Ranked queue created.


,rank,client_hash_id,content_hash_id,action_score,reason_code,action,impressions,ctr,avg_position
0,1,client_3197e6291363b4db,content_65b8a4998e633d89,0.864966,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,8905,0.0,68.385294
1,2,client_23a62021009f63c4,content_295e883e0e86ca3c,0.860429,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,21939,0.0,51.573254
2,3,client_e547b89c05043229,content_4002467a580a7f98,0.859251,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,11973,0.0,54.584394
3,4,client_23a62021009f63c4,content_bbf8e4d669f253cf,0.857517,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,28934,0.0,47.084829
4,5,client_23a62021009f63c4,content_959d535a9fcc865c,0.857138,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,10452,0.0,53.183798
5,6,client_23a62021009f63c4,content_066bb7aeff9aeea8,0.855728,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,11443,0.0,50.371982
6,7,client_e547b89c05043229,content_6177aad2ded9dee5,0.854986,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,10333,0.0,50.566177
7,8,client_23a62021009f63c4,content_421ad263ab8c8c03,0.854110,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,10190,0.0,49.663625
8,9,client_23a62021009f63c4,content_2da022341f8803c3,0.853783,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,15285,0.0,45.952191
9,10,client_23a62021009f63c4,content_99ca372d2f17bd96,0.853342,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,12567,0.0,46.871729


In [8]:
output_path = "work/outputs/action_playbook_queue.csv"

playbook_df.to_csv(
    output_path,
    index=False
)

print("Saved successfully:", output_path)

Saved successfully: work/outputs/action_playbook_queue.csv


In [10]:
print("Number of recommendations:", len(playbook_df))

print("\nReason codes:")
print(playbook_df["reason_code"].value_counts())

print("\nActions:")
print(playbook_df["action"].value_counts())

display(playbook_df.head(10))

Number of recommendations: 331437

Reason codes:
reason_code
LOW_CTR          262600
REVIEW            52792
WEAK_POSITION     16045
Name: count, dtype: int64

Actions:
action
PRIORITIZE_FOR_HUMAN_REVIEW    331437
Name: count, dtype: int64


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,impressions_rank,ctr_opportunity,position_opportunity,action_score,reason_code,action,rank
0,client_3197e6291363b4db,content_65b8a4998e633d89,8905,0,68.385294,0.0,0.979354,0.603845,0.973571,0.864966,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,1
1,client_23a62021009f63c4,content_295e883e0e86ca3c,21939,0,51.573254,0.0,0.994724,0.603845,0.937953,0.860429,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,2
2,client_e547b89c05043229,content_4002467a580a7f98,11973,0,54.584394,0.0,0.986151,0.603845,0.945456,0.859251,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,3
3,client_23a62021009f63c4,content_bbf8e4d669f253cf,28934,0,47.084829,0.0,0.996844,0.603845,0.925421,0.857517,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,4
4,client_23a62021009f63c4,content_959d535a9fcc865c,10452,0,53.183798,0.0,0.983330,0.603845,0.942174,0.857138,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,5
5,client_23a62021009f63c4,content_066bb7aeff9aeea8,11443,0,50.371982,0.0,0.985288,0.603845,0.934864,0.855728,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,6
6,client_e547b89c05043229,content_6177aad2ded9dee5,10333,0,50.566177,0.0,0.983042,0.603845,0.935385,0.854986,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,7
7,client_23a62021009f63c4,content_421ad263ab8c8c03,10190,0,49.663625,0.0,0.982740,0.603845,0.932867,0.854110,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,8
8,client_23a62021009f63c4,content_2da022341f8803c3,15285,0,45.952191,0.0,0.990309,0.603845,0.921686,0.853783,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,9
9,client_23a62021009f63c4,content_99ca372d2f17bd96,12567,0,46.871729,0.0,0.987094,0.603845,0.924504,0.853342,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,10


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*


### Intended use

The queue is intended for SEO and content teams to decide which pages should be reviewed first.

The output supports prioritization by highlighting pages with observed search-performance patterns that may deserve attention.

### Limits

The score does not prove that a content refresh will improve traffic, ranking, or CTR.

It should not be interpreted as a prediction of Google's ranking system.

The recommendations are based on the available anonymized data and may not capture content quality, business importance, seasonality, or other factors that are not represented in the dataset.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every recommendation should be reviewed by a human before any content change is made.

A reviewer should check:

- Whether the page is still relevant to its intended search intent.
- Whether the low CTR is reasonable for the page's position and query context.
- Whether the content is already accurate and complete.
- Whether there are business or editorial reasons not visible in the dataset.

### What should NOT be automated

The system should not automatically:

- Rewrite or delete content.
- Change titles or metadata without review.
- Decide that a page is low quality.
- Publish content changes.
- Treat a high score as proof that a refresh will improve performance.

The model and baseline should remain decision-support tools rather than autonomous content editors.

In [11]:
review_sample = playbook_df[
    [
        "rank",
        "content_hash_id",
        "action_score",
        "reason_code",
        "action",
        "impressions",
        "ctr",
        "avg_position"
    ]
].head(10)

display(review_sample)

,rank,content_hash_id,action_score,reason_code,action,impressions,ctr,avg_position
0,1,content_65b8a4998e633d89,0.864966,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,8905,0.0,68.385294
1,2,content_295e883e0e86ca3c,0.860429,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,21939,0.0,51.573254
2,3,content_4002467a580a7f98,0.859251,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,11973,0.0,54.584394
3,4,content_bbf8e4d669f253cf,0.857517,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,28934,0.0,47.084829
4,5,content_959d535a9fcc865c,0.857138,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,10452,0.0,53.183798
5,6,content_066bb7aeff9aeea8,0.855728,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,11443,0.0,50.371982
6,7,content_6177aad2ded9dee5,0.854986,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,10333,0.0,50.566177
7,8,content_421ad263ab8c8c03,0.854110,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,10190,0.0,49.663625
8,9,content_2da022341f8803c3,0.853783,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,15285,0.0,45.952191
9,10,content_99ca372d2f17bd96,0.853342,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,12567,0.0,46.871729


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*


The recommendations should be monitored over time because search behavior and content performance can change.

Useful warning signs include:

- The distribution of CTR or average position changes substantially.
- The proportion of pages receiving each reason code changes unexpectedly.
- The baseline or model performance decreases on newly observed data.
- The relationship between the input signals and review outcomes changes.

A retraining or rule-review process should be considered when these changes persist rather than after a single unusual period.

These triggers are monitoring signals, not automatic retraining decisions.

In [12]:
print("Current queue size:", len(playbook_df))

print("\nScore summary:")
display(playbook_df["action_score"].describe())

print("\nReason-code distribution:")
display(playbook_df["reason_code"].value_counts(normalize=True).rename("proportion"))

Current queue size: 331437

Score summary:


,action_score
count,176738.000000
mean,0.566082
std,0.113564
min,0.194142
25%,0.476928
50%,0.558320
75%,0.661106
max,0.864966



Reason-code distribution:


,proportion
reason_code,
LOW_CTR,0.792307
REVIEW,0.159282
WEAK_POSITION,0.048410


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*


The ranked action queue is exported to `work/outputs/` so that it can be reused in the final paper.

The exported queue contains the ranking, action score, reason code, action, and supporting observed signals.

In [13]:
import os

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/action_playbook_queue.csv"

playbook_df.to_csv(output_path, index=False)

print("Exported:", output_path)
print("Rows:", len(playbook_df))

Exported: work/outputs/action_playbook_queue.csv
Rows: 331437


In [14]:


check = pd.read_csv("work/outputs/action_playbook_queue.csv")

print("Export verified.")
print("Shape:", check.shape)

display(check.head(10))

Export verified.
Shape: (331437, 13)


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,impressions_rank,ctr_opportunity,position_opportunity,action_score,reason_code,action,rank
0,client_3197e6291363b4db,content_65b8a4998e633d89,8905,0,68.385294,0.0,0.979354,0.603845,0.973571,0.864966,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,1
1,client_23a62021009f63c4,content_295e883e0e86ca3c,21939,0,51.573254,0.0,0.994724,0.603845,0.937953,0.860429,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,2
2,client_e547b89c05043229,content_4002467a580a7f98,11973,0,54.584394,0.0,0.986151,0.603845,0.945456,0.859251,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,3
3,client_23a62021009f63c4,content_bbf8e4d669f253cf,28934,0,47.084829,0.0,0.996844,0.603845,0.925421,0.857517,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,4
4,client_23a62021009f63c4,content_959d535a9fcc865c,10452,0,53.183798,0.0,0.983330,0.603845,0.942174,0.857138,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,5
5,client_23a62021009f63c4,content_066bb7aeff9aeea8,11443,0,50.371982,0.0,0.985288,0.603845,0.934864,0.855728,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,6
6,client_e547b89c05043229,content_6177aad2ded9dee5,10333,0,50.566177,0.0,0.983042,0.603845,0.935385,0.854986,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,7
7,client_23a62021009f63c4,content_421ad263ab8c8c03,10190,0,49.663625,0.0,0.982740,0.603845,0.932867,0.854110,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,8
8,client_23a62021009f63c4,content_2da022341f8803c3,15285,0,45.952191,0.0,0.990309,0.603845,0.921686,0.853783,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,9
9,client_23a62021009f63c4,content_99ca372d2f17bd96,12567,0,46.871729,0.0,0.987094,0.603845,0.924504,0.853342,LOW_CTR,PRIORITIZE_FOR_HUMAN_REVIEW,10


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.